In [57]:
# Load env variables and create client
from dotenv import load_dotenv
import os 
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.environ["CLAUDE_MODEL"] 

In [58]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [75]:
import json

def generate_dataset():  
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [ 
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex", 
                "solution_criteria": "key criteria for evaluating the solution" 
            },
            ...additional 
        ]
        ```


        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json") 
    text = chat(messages, stop_sequences= ["```"]) 
    return json.loads(text) 

In [76]:
dataset = generate_dataset() 
dataset

[{'task': "Create an IAM policy that allows read-only access to a specific S3 bucket named 'company-logs-2024' and all objects within it",
  'format': 'json',
  'solution_criteria': 'Policy must include s3:GetObject and s3:ListBucket actions, specify the correct bucket ARN in Resource field, and follow proper IAM policy JSON structure with Version and Statement elements'},
 {'task': "Write a Python function that takes a list of EC2 instance dictionaries (with 'InstanceId' and 'State' keys) and returns only the instance IDs of instances in 'running' state",
  'format': 'python',
  'solution_criteria': "Function should filter instances by State=='running', extract InstanceId values, return a list of strings, and handle empty input gracefully"},
 {'task': 'Write a regular expression that validates AWS ARN format for S3 buckets (e.g., arn:aws:s3:::bucket-name or arn:aws:s3:::bucket-name/key)',
  'format': 'regex',
  'solution_criteria': "Regex must match the pattern 'arn:aws:s3:::' followe

In [77]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2) 

In [78]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]} 

        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
    """
    
    messages = [] 
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")  
    output = chat(messages, stop_sequences= ["```"]) 
    return output 

In [81]:
# Graders:
#     1. Code - Programatically evaluate the result 
#     2. Model - Ask a model to assign a score to the output or compare two versions
#     3. Human - Ask a Human to assign a score to the output, or compare two versions      


def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        
        Task: {test_case["task"]} 
        Solution: {output}
        Criteria you should use to evaluate the solution: {test_case["solution_criteria"]}
        
        Provide your evaluation as a structured JSON object with:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement  
        - "reasoning": A concise explanation of your assessment
        - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text) 


# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [82]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case) 

    score = (score + syntax_score)/2   
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning 
    }

In [83]:
from statistics import mean 
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results 

In [84]:
with open("dataset.json", "r") as f:
    dataset = json.load(f) 

results = run_eval(dataset) 

Average score: 8.666666666666666


In [85]:
results 

[{'output': '\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Effect": "Allow",\n      "Action": [\n        "s3:GetObject",\n        "s3:GetObjectVersion",\n        "s3:ListBucket",\n        "s3:GetBucketLocation",\n        "s3:GetBucketVersioning"\n      ],\n      "Resource": [\n        "arn:aws:s3:::company-logs-2024",\n        "arn:aws:s3:::company-logs-2024/*"\n      ]\n    }\n  ]\n}\n',
  'test_case': {'task': "Create an IAM policy that allows read-only access to a specific S3 bucket named 'company-logs-2024' and all objects within it",
   'format': 'json',
   'solution_criteria': 'Policy must include s3:GetObject and s3:ListBucket actions, specify the correct bucket ARN in Resource field, and follow proper IAM policy JSON structure with Version and Statement elements'},
  'score': 9.5,
  'reasoning': "The solution successfully meets all the core requirements: it includes the mandatory s3:GetObject and s3:ListBucket actions, specifies the correct bucket ARN in the 

In [86]:
print(json.dumps(results, indent = 2)) 

[
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:GetObjectVersion\",\n        \"s3:ListBucket\",\n        \"s3:GetBucketLocation\",\n        \"s3:GetBucketVersioning\"\n      ],\n      \"Resource\": [\n        \"arn:aws:s3:::company-logs-2024\",\n        \"arn:aws:s3:::company-logs-2024/*\"\n      ]\n    }\n  ]\n}\n",
    "test_case": {
      "task": "Create an IAM policy that allows read-only access to a specific S3 bucket named 'company-logs-2024' and all objects within it",
      "format": "json",
      "solution_criteria": "Policy must include s3:GetObject and s3:ListBucket actions, specify the correct bucket ARN in Resource field, and follow proper IAM policy JSON structure with Version and Statement elements"
    },
    "score": 9.5,
    "reasoning": "The solution successfully meets all the core requirements: it includes the mandatory s3:GetObject and s3: